# 01 - exploración

**Objetivo:** Cargar `train.csv` de la competencia y realizar una exploración básica para validar lectura, comprender dimensiones, variables clave y distribución de la(s) etiqueta(s).

> Ejecuta este notebook en el mismo directorio donde se ubique `data/train.csv` o ajusta la ruta en la celda de carga.


In [ ]:
# Imports
import os, re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 120)


In [ ]:
# Rutas
DATA_DIR = "data"
TRAIN_PATH = os.path.join(DATA_DIR, "train.csv")

assert os.path.exists(TRAIN_PATH), f"No se encontró {TRAIN_PATH}. Coloca 'train.csv' en la carpeta 'data/'."


In [ ]:
# Carga de datos
df = pd.read_csv(TRAIN_PATH)
print("Dimensiones:", df.shape)
display(df.head())

print("\nTipos de datos:")
display(df.dtypes)

print("\nValores faltantes (conteo):")
display(df.isna().sum().sort_values(ascending=False).head(20))


In [ ]:
# Intento de detección de columna de etiqueta (target) por nombre
candidatos = [c for c in df.columns if re.search(r'(target|clase|class|label|y)$', c, flags=re.IGNORECASE)]
target_col = candidatos[0] if len(candidatos) else None
print("Posible columna de destino (target):", target_col)


In [ ]:
# Distribución de clases (si existe target)
if target_col is not None:
    vc = df[target_col].value_counts(dropna=False)
    print("Distribución de", target_col)
    display(vc)
    
    # Gráfico de barras (matplotlib, sin estilos ni colores personalizados)
    plt.figure()
    vc.plot(kind="bar")
    plt.title(f"Distribución de {target_col}")
    plt.xlabel(target_col)
    plt.ylabel("Frecuencia")
    plt.tight_layout()
    plt.show()
else:
    print("No se identificó automáticamente una columna de etiqueta. Revisa el nombre en el dataset.")


In [ ]:
# Exploración por 'localidad' si hay una columna relacionada
cand_localidad = [c for c in df.columns if re.search(r'(local|municipio|ciudad|region|departamento)', c, flags=re.IGNORECASE)]
loc_col = cand_localidad[0] if len(cand_localidad) else None
print("Posible columna de localidad:", loc_col)

if loc_col is not None:
    conteo_loc = df[loc_col].value_counts().head(20)
    print("Top 20 localidades por frecuencia:")
    display(conteo_loc)
    
    plt.figure()
    conteo_loc.plot(kind="bar")
    plt.title(f"Top 20 {loc_col} por frecuencia")
    plt.xlabel(loc_col)
    plt.ylabel("Conteo")
    plt.tight_layout()
    plt.show()
    
    if 'target_col' in locals() and target_col is not None:
        # tabla cruzada
        ct = pd.crosstab(df[loc_col], df[target_col])
        display(ct.head(10))
else:
    print("No se encontró columna de localidad. Si existe con otro nombre, ajústalo manualmente.")


In [ ]:
# Estadísticos generales
display(df.describe(include='all').transpose().head(30))


In [ ]:
# Guarda una muestra pequeña para revisar estructura sin subir datos sensibles al repositorio
sample_path = os.path.join(DATA_DIR, "train_sample_100.csv")
df.sample(min(100, len(df)), random_state=42).to_csv(sample_path, index=False)
print("Muestra guardada en:", sample_path)
